In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [2]:
df = sns.load_dataset('iris')

df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [3]:
X = df.drop('species', axis=1)
y = df['species']

In [4]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [6]:


base_learners = [  
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('svc', SVC(probability=True, kernel='rbf', random_state=42)),  # Added equals sign to random_state=42
    ('lr', LogisticRegression(max_iter=1000))
]



In [7]:
meta_learner = LogisticRegression(max_iter=1000)

In [8]:
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator= meta_learner,
    cv=5
)



In [9]:
stacking_clf.fit(X_train,y_train)

StackingClassifier(cv=5,
                   estimators=[('dt', DecisionTreeClassifier(random_state=42)),
                               ('svc', SVC(probability=True, random_state=42)),
                               ('lr', LogisticRegression(max_iter=1000))],
                   final_estimator=LogisticRegression(max_iter=1000))

In [10]:
y_pred = stacking_clf.predict(X_test)

In [11]:
accuracy = accuracy_score(y_test,y_pred)
accuracy

0.9666666666666667

In [ ]:
# point should be noted--- when we use grid search we not use train test split but in ensemble learning (stacking) case we 
# have to use train test split and because data is divided in 2 parts 80 percent and 20 percent  so we do testing in 80percent so 20 percent of the
# data is still unseen
# that why we use train test split in here



# Note:
# - Grid search (with cross-validation) is normally run on the training data to tune hyperparameters.
# - For stacking/ensemble models we still keep a separate hold-out test set (e.g. 20%) so the final evaluation
#   reflects performance on truly unseen data.
# - During stacking, base learners are trained using cross-validated / out-of-fold predictions on the training
#   portion to create inputs for the meta-learner; the hold-out test set must remain unseen until the final eval.
# Therefore: first do train_test_split (e.g. 80% train / 20% test), run CV/grid-search and stacking procedures
# only on the training data, and use the reserved 20% as the final test set.

In [12]:
from sklearn.ensemble import RandomForestClassifier


In [15]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)

In [16]:
rf_model.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

In [17]:
y_pred = rf_model.predict(X_test)
y_pred

array([0, 2, 1, 1, 0, 1, 0, 0, 2, 1, 2, 2, 2, 1, 0, 0, 0, 1, 1, 1, 0, 2,
       1, 1, 2, 2, 1, 0, 2, 0])

In [19]:
accuracy = accuracy_score(y_test,y_pred)
accuracy

0.9

In [ ]:
# xg boost , adaboost , gradientboost

In [20]:
from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier

In [21]:
ada_model = AdaBoostClassifier(n_estimators=100,random_state=42)
ada_model.fit(X_train,y_train)

AdaBoostClassifier(n_estimators=100, random_state=42)

In [30]:
Y_pred_ada = ada_model.predict(X_test)
accuracy = accuracy_score(y_test,Y_pred_ada)
accuracy

0.9333333333333333

In [ ]:
from typing import Tuple, Any
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier


def create_sample_dataset() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Creates a sample classification dataset for demonstration.
    
    Returns:
        Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]: 
            Training and testing features and labels (X_train, X_test, y_train, y_test)
    """
    X, y = make_classification(
        n_samples=1000, 
        n_features=20, 
        n_informative=10, 
        n_redundant=10, 
        random_state=42
    )
    return train_test_split(X, y, test_size=0.2, random_state=42)


def demonstrate_adaboost(X_train: np.ndarray, X_test: np.ndarray, 
                        y_train: np.ndarray, y_test: np.ndarray) -> float:
    """
    Demonstrates AdaBoost classifier usage.
    
    Args:
        X_train (np.ndarray): Training features
        X_test (np.ndarray): Testing features
        y_train (np.ndarray): Training labels
        y_test (np.ndarray): Testing labels
    
    Returns:
        float: Accuracy score of the model
    """
    ada_classifier = AdaBoostClassifier(
        n_estimators=100,
        learning_rate=1.0,
        random_state=42
    )
    ada_classifier.fit(X_train, y_train)
    predictions = ada_classifier.predict(X_test)
    return accuracy_score(y_test, predictions)


def demonstrate_gradient_boosting(X_train: np.ndarray, X_test: np.ndarray, 
                                 y_train: np.ndarray, y_test: np.ndarray) -> float:
    """
    Demonstrates Gradient Boosting classifier usage.
    
    Args:
        X_train (np.ndarray): Training features
        X_test (np.ndarray): Testing features
        y_train (np.ndarray): Training labels
        y_test (np.ndarray): Testing labels
    
    Returns:
        float: Accuracy score of the model
    """
    gb_classifier = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
    gb_classifier.fit(X_train, y_train)
    predictions = gb_classifier.predict(X_test)
    return accuracy_score(y_test, predictions)


def demonstrate_xgboost(X_train: np.ndarray, X_test: np.ndarray, 
                       y_train: np.ndarray, y_test: np.ndarray) -> float:
    """
    Demonstrates XGBoost classifier usage.
    
    Args:
        X_train (np.ndarray): Training features
        X_test (np.ndarray): Testing features
        y_train (np.ndarray): Training labels
        y_test (np.ndarray): Testing labels
    
    Returns:
        float: Accuracy score of the model
    """
    xgb_classifier = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
    xgb_classifier.fit(X_train, y_train)
    predictions = xgb_classifier.predict(X_test)
    return accuracy_score(y_test, predictions)


def main() -> None:
    """
    Main function to demonstrate all three ensemble classifiers.
    """
    # Create sample dataset
    X_train, X_test, y_train, y_test = create_sample_dataset()
    
    # Demonstrate each classifier
    ada_accuracy = demonstrate_adaboost(X_train, X_test, y_train, y_test)
    gb_accuracy = demonstrate_gradient_boosting(X_train, X_test, y_train, y_test)
    xgb_accuracy = demonstrate_xgboost(X_train, X_test, y_train, y_test)
    
    # Print results
    print(f"AdaBoost Classifier Accuracy: {ada_accuracy:.4f}")
    print(f"Gradient Boosting Classifier Accuracy: {gb_accuracy:.4f}")
    print(f"XGBoost Classifier Accuracy: {xgb_accuracy:.4f}")


if __name__ == "__main__":
    main()